# 04 — Runout (com4FlowPy)

**Time:** 13:50–14:50 · BFW-led

Typical alpha **30°**, infrequent **18°**. Flow-Py is mass-routing, not RAMMS.

On this 13 km², ~21 m grid, both scenarios finished in **~3.5 s** on a desktop
with the numba engine. If a laptop run is still going with 15 minutes left in
the block, load the reference rasters so we can look at the maps together.
Please keep this block on the Connaught 30 m clip, not the 5 m lidar or South Coast.


In [ ]:
from pathlib import Path
import os, sys
sys.path.insert(0, str(Path(".").resolve()))
from workshop import REPO, DATA, OUT, need, read_raster, show
import matplotlib.pyplot as plt

os.chdir(REPO)
pra_dir = OUT / "PRA"
sieve_f = need(pra_dir / "pra_frequent_pra_sieve.tif",
               DATA / "03_pra" / "outputs_reference" / "pra_frequent_pra_sieve.tif")
sieve_e = need(pra_dir / "pra_extreme_pra_sieve.tif",
               DATA / "03_pra" / "outputs_reference" / "pra_extreme_pra_sieve.tif")
dem = need(pra_dir / "autoATES_v3_DSM_00.tif",
           DATA / "01_elevation" / "outputs_reference" / "dem_aligned.tif")
print("release rasters ok")

In [ ]:
from autoates.comAutoATES.comAutoATES import load_config
from autoates.comAutoATES.runout.runoutCore import run_flowpy

cfg = load_config(REPO / "config" / "autoATESCfg_workshop.ini")
flowpy_dir = OUT / "FlowPy"
runout = {}
for scen, release, forest in (
    ("frequent", sieve_f, pra_dir / "pra_frequent_forest_scale.tif"),
    ("extreme", sieve_e, pra_dir / "pra_extreme_forest_scale.tif"),
):
    forest = need(forest, DATA / "03_pra" / "outputs_reference" / forest.name)
    print("running", scen, "alpha", cfg["FlowPy"][f"{scen}_alpha"])
    runout[scen] = run_flowpy(
        dem_path=dem, release_path=release, scenario=scen,
        config=cfg, output_dir=flowpy_dir, forest_path=forest,
    )
    print(" ", scen, "zDelta", Path(runout[scen]["zDelta"]).name)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 5))
for ax, scen, title in (
    (axs[0], "frequent", "Typical reach (α 30°)"),
    (axs[1], "extreme", "Infrequent reach (α 18°)"),
):
    tifs = list((OUT / "FlowPy" / scen).glob("res_*/com4_*_run_fpTravelAngleMax.tif"))
    if not tifs:
        tifs = list((DATA / "04_runout" / "outputs_reference" / scen).glob("*fpTravelAngleMax.tif"))
    arr, *_ = read_raster(tifs[-1])
    show(arr, title, "plasma", ax=ax)
plt.tight_layout()

Travel angle is *reach*. Also look at `zdelta` (energy-like) and `routFluxSum`
(mass routing) in the same `res_*` folder, or in QGIS.

**If stuck:** `data/04_runout/outputs_reference/frequent/` and `.../extreme/`.
